In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Check if hazm is installed
try:
    from hazm import Normalizer, word_tokenize, stopwords_list
    print("✅ hazm library loaded successfully")
except ImportError:
    print("❌ hazm not found. Please install it: pip install hazm")

print("✅ All libraries loaded")


✅ hazm library loaded successfully
✅ All libraries loaded


In [2]:
# Load the data from feature engginiering step
df = pd.read_csv('../data/processed/features_step2.csv')

print(f"Total records: {len(df):,}")
print(f"\nAvailable columns:")
print(df.columns.tolist())

Total records: 100,000

Available columns:
['id_comment', 'title', 'body', 'created_at', 'rate', 'recommendation_status', 'is_buyer', 'product_id', 'advantages', 'disadvantages', 'likes', 'dislikes', 'seller_title', 'seller_code', 'true_to_size_rate', 'id_product', 'title_fa', 'Rate', 'Rate_cnt', 'Category1', 'Category2', 'Brand', 'Price', 'Seller', 'Is_Fake', 'min_price_last_month', 'sub_category', 'full_text', 'text_length', 'body_length', 'title_length', 'word_count', 'has_advantages', 'has_disadvantages', 'sentiment_label', 'price_change_pct', 'has_product_info', 'product_popularity', 'seller_avg_rate', 'seller_comment_count', 'seller_rate_std', 'seller_total_likes', 'seller_total_dislikes', 'seller_like_ratio']


In [10]:
from hazm import Normalizer, word_tokenize, stopwords_list
import re

# Initialize normalizer
normalizer = Normalizer()

# Get Persian stopwords
# persian_stopwords = set(stopwords_list())

# Remove sentiment-related words from stopwords
# sentiment_words = {
#     'خوب', 'بد', 'عالی', 'مناسب', 'ضعیف', 'بهترین', 'بدترین',
#     'خیلی', 'زیاد', 'کم', 'نه', 'نمی', 'نیست'
# }
# stop_words = persian_stopwords - sentiment_words

def preprocess_persian_text(text):
    """
    Preprocess Persian text:
    - Normalize
    - Remove extra spaces
    - Remove English characters and numbers
    - Tokenize
    - Remove stopwords
    - Remove short words (< 2 chars)
    """
    if pd.isna(text) or text == '':
        return ''
    
    # Normalize text
    text = normalizer.normalize(str(text))
    
    # Remove English characters and numbers
    text = re.sub(r'[a-zA-Z0-9]+', ' ', text)
    
    # Remove extra punctuation
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and short words
    # tokens = [word for word in tokens if word not in persian_stopwords and len(word) > 1]
    
    return ' '.join(tokens)

# Test the function
sample_text = df['full_text'].iloc[22]
print("Original text:")
print(sample_text[:200])
print("\n" + "="*50 + "\n")
print("Preprocessed text:")
print(preprocess_persian_text(sample_text)[:200])


Original text:
نسبت به قیمتی که پرداخت میکنید خوبه فقط شانه برس خیلی زبر می باشد در کل نسبت به پولی که پرداخت میکنید خوبه ['ارزان'] ['شانه های برس زبر می باشد']


Preprocessed text:
نسبت به قیمتی که پرداخت می کنید خوبه فقط شانه برس خیلی زبر می باشد در کل نسبت به پولی که پرداخت می کنید خوبه ارزان شانه های برس زبر می باشد


In [12]:
# Apply preprocessing to full_text column
print("Preprocessing full_text column...")
df['full_text_cleaned'] = df['full_text'].apply(preprocess_persian_text)

# Apply preprocessing to title column
print("Preprocessing title column...")
df['title_cleaned'] = df['title'].apply(preprocess_persian_text)

# Apply preprocessing to body column
print("Preprocessing body column...")
df['body_cleaned'] = df['body'].apply(preprocess_persian_text)

# Check results
print("\nPreprocessing completed!")
print(f"Dataset shape: {df.shape}")
print("\nSample comparison:")
print("="*80)
print("Original full_text:")
print(df['full_text'].iloc[5])
print("\nCleaned full_text:")
print(df['full_text_cleaned'].iloc[5])

# Save preprocessed data
output_file = '../data/processed/preprocessed_data.csv'
print(f"\n{'='*80}")
print(f"Saving preprocessed data to {output_file}...")
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n✅ Successfully saved {len(df):,} records!")
print(f"\nNew columns added:")
print("  - full_text_cleaned")
print("  - title_cleaned") 
print("  - body_cleaned")
print("  - cleaned_length")
print("  - cleaned_word_count")

print(f"\n{'='*80}")
print("Ready for EDA! 🚀")
print(f"{'='*80}")

Preprocessing full_text column...
Preprocessing title column...
Preprocessing body column...

Preprocessing completed!
Dataset shape: (100000, 47)

Sample comparison:
Original full_text:
خوبه قبلا هم استفاده کردم اگه بلد باشین کار کردن باهاش رو 
برای ی پاکسازی پوست سطحی توی خونه عالیه

Cleaned full_text:
خوبه قبلا هم استفاده کردم اگه بلد باشین کار کردن باهاش رو برای ی پاکسازی پوست سطحی توی خونه عالیه

Saving preprocessed data to ../data/processed/preprocessed_data.csv...

✅ Successfully saved 100,000 records!

New columns added:
  - full_text_cleaned
  - title_cleaned
  - body_cleaned
  - cleaned_length
  - cleaned_word_count

Ready for EDA! 🚀
